# Практическая работа №1
## Линейная регрессия на данных Auto MPG

**Цель работы** — применение линейной регрессии для прогнозирования топливной экономичности автомобиля (`mpg`) по его техническим характеристикам.

В работе был использован набор данных Auto MPG. Были выполнены проверка и очистка таблицы, анализ распределений признаков и их связи с `mpg`. После подготовки данных была обучена линейная регрессия, а качество прогноза было оценено по метрикам MAE, RMSE и R². Таблицы и графики позволяют рассмотреть результаты предсказаний и характер ошибок.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scipy scikit-learn

Для работы были выбраны библиотеки `pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy` и `scikit-learn`. Команда `%pip install` устанавливает их в среду, в которой открыт ноутбук. Параметр `-q` уменьшает количество служебных сообщений при установке.

После запуска ячейки могут появиться сообщения об установке пакетов. Числовых результатов анализа на данном этапе ещё нет: библиотеки только подготавливаются к дальнейшей работе.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
)

# В браузерной среде используется open_url, в локальной — pandas.
try:
    from pyodide.http import open_url
except ImportError:
    open_url = None

from scipy import stats
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (precision_recall_curve, average_precision_score, roc_curve)
from sklearn.calibration import calibration_curve


Были подключены инструменты для обработки таблиц, вычислений, визуализации и обучения моделей. Для работы с файлами используется `Path`, таблицами — `pandas`, массивами — `numpy`. Графики строятся средствами `matplotlib` и `seaborn`. Из `scikit-learn` импортированы алгоритмы, функции разделения выборки, преобразования признаков и метрики качества. `scipy.stats` применяется для статистических графиков.

Отдельный блок `try / except ImportError` проверяет возможность загрузки файлов через `pyodide.http.open_url`. Это сделано для совместимости с браузерным Jupyter. В обычном Python переменная `open_url` остаётся равной `None`. Если все импорты выполнены без ошибок, ячейка ничего не печатает, но все указанные функции становятся доступны далее.

In [ ]:
def read_csv_compat(url, local_path=None, **read_csv_kwargs):
    if local_path is not None:
        local_path = Path(local_path)
        if local_path.exists():
            return pd.read_csv(local_path, **read_csv_kwargs)

    if open_url is not None:
        data = pd.read_csv(open_url(url), **read_csv_kwargs)
    else:
        data = pd.read_csv(url, **read_csv_kwargs)

    if local_path is not None:
        local_path.parent.mkdir(parents=True, exist_ok=True)
        data.to_csv(local_path, index=False)

    return data


Была задана функция `read_csv_compat`, которая используется при загрузке всех четырёх наборов данных. Сначала проверяется наличие локального CSV. Если он уже сохранён, таблица читается с диска. При отсутствии файла выполняется загрузка по URL: в браузерной среде через `open_url`, в обычном Jupyter через `pd.read_csv`. После загрузки данные записываются в папку `data`.

Аргумент `**read_csv_kwargs` позволяет при необходимости передавать дополнительные настройки чтения CSV. Команда `return data` возвращает подготовленную таблицу `DataFrame`. На экране после объявления функции ничего не появляется: фактическое чтение будет выполнено при её вызове.

## 1. Загрузка данных

Для анализа использовался набор данных Auto MPG. Таблица содержит характеристики автомобилей и показатель `mpg`, который принят в качестве целевой переменной.

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

data_path = DATA_DIR / "auto_mpg.csv"
data_url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv"

df = read_csv_compat(data_url, data_path)
df = df.rename(columns={"name": "car_name"})

df.head()


Для набора Auto MPG был указан URL таблицы с характеристиками автомобилей. Папка `data` создаётся через `Path(...).mkdir(exist_ok=True)`, после чего CSV считывается функцией `read_csv_compat` в `df`. Команда `rename(columns={"name": "car_name"})` делает название текстового поля более понятным. В данном CSV заголовки столбцов уже присутствуют, поэтому повторно задавать их не требуется.

Результат `df.head()` — первые пять автомобилей с характеристиками и целевым столбцом `mpg`. По этому фрагменту проверяется правильность чтения и наличие необходимых признаков. Значение `mpg` показывает, сколько миль автомобиль проезжает на одном галлоне топлива; большая величина соответствует лучшей топливной экономичности.

## 2. Первичный анализ

In [ ]:
print("Размер датасета:", df.shape)
display(df.head())
df.info()

Для первичной проверки были использованы `df.shape`, `df.head()` и `df.info()`. Первая команда показывает размер набора данных в виде `(число строк, число столбцов)`. Вторая выводит небольшой фрагмент таблицы. Третья показывает тип данных каждого столбца и количество непустых значений (`Non-Null Count`).

Если для какого-либо признака число непустых значений меньше общего числа строк, в нём имеются пропуски. Также по `dtype` можно проверить, что числовые признаки действительно распознаны как числа, а названия категорий — как текст.

In [ ]:
df.describe(include='all').T

Была получена описательная статистика с помощью `describe()`. Показатель `count` отражает число заполненных значений; `mean` — среднее; `std` — стандартное отклонение; `min` и `max` — границы наблюдаемого диапазона. Квартили `25%`, `50%` и `75%` характеризуют положение основной части данных, при этом `50%` соответствует медиане. Операция `.T` разворачивает таблицу, чтобы признаки располагались строками.

По полученной таблице удобно сравнивать диапазоны значений. Большая разница между средним и медианой может говорить об асимметрии распределения, а крайние минимумы и максимумы требуют дополнительной проверки.

## 3. Проверка качества данных

Перед обучением была выполнена проверка пропусков, дубликатов и нетипичных значений в числовых признаках.

In [ ]:
print("Пропуски по столбцам:")
display(df.isnull().sum().to_frame("missing"))

print("Количество полных дубликатов:", df.duplicated().sum())

Проверка пропусков была выполнена выражением `df.isnull().sum()`: оно подсчитывает отсутствующие значения для каждого столбца. Отдельно командой `df.duplicated().sum()` определяется число полностью совпадающих строк. Таблица `missing` содержит количество пропусков, а следующая строка вывода — число дубликатов.

Нулевые значения в этих результатах означают, что соответствующая проблема не обнаружена. Ненулевые значения показывают, какие столбцы или строки требуют обработки. Для задач классификации дополнительно выводится количество объектов каждого класса: это позволяет оценить их представленность ещё до обучения.

In [ ]:
numeric_columns = [
    "mpg", "cylinders", "displacement", "horsepower",
    "weight", "acceleration", "model_year"
]

outliers = {}
for column in numeric_columns:
    values = df[column].dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    outliers[column] = ((values < low) | (values > high)).sum()

pd.Series(outliers, name="Количество значений за границами IQR")

Был выполнен поиск необычных значений в семи числовых столбцах: `mpg`, `cylinders`, `displacement`, `horsepower`, `weight`, `acceleration` и `model_year`. В цикле по каждому признаку вычисляются квартили `Q1` и `Q3`, затем межквартильный размах `IQR`. Значения ниже `Q1 - 1.5 × IQR` и выше `Q3 + 1.5 × IQR` учитываются как возможные выбросы.

Результат выводится в виде списка признаков с количеством таких наблюдений. Это не обязательно некорректные записи: например, мощный или тяжёлый автомобиль может сильно отличаться от большинства, но оставаться реальным объектом. По этой причине одних результатов IQR недостаточно для удаления строки.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 8))
axes = axes.ravel()

for index, column in enumerate(numeric_columns):
    axes[index].boxplot(df[column].dropna())
    axes[index].set_title(column)
    axes[index].set_ylabel("Значение")

for index in range(len(numeric_columns), len(axes)):
    axes[index].axis("off")

plt.suptitle("Диаграммы размаха числовых признаков", y=1.02)
plt.tight_layout()
plt.show()

Для каждого числового признака построен отдельный boxplot, поэтому признаки с разными масштабами не мешают друг другу. Коробка показывает промежуток между первым и третьим квартилями, линия внутри коробки — медиану, «усы» — основной диапазон значений, а отдельные точки за усами являются потенциальными выбросами.

График помогает заметить необычно большие или маленькие значения. Такие точки не удаляются автоматически: например, автомобиль действительно может иметь необычно высокую мощность или массу. Перед исключением таких наблюдений необходимо проверить, являются ли они ошибками измерения.

Диаграммы построены раздельно, поскольку вес автомобиля измеряется тысячами фунтов, а другие признаки имеют значительно меньшие масштабы. Общая ось Y скрыла бы их разброс.

In [ ]:
before = len(df)
df = df.drop_duplicates().dropna(subset=["horsepower"]).copy()

print("Строк до очистки:", before)
print("Строк после очистки:", len(df))
print("Оставшиеся пропуски:", int(df.isnull().sum().sum()))

На этапе очистки сначала было сохранено исходное число строк `before`. Затем `drop_duplicates()` удаляет полностью совпадающие записи, а `dropna(subset=["horsepower"])` исключает автомобили, для которых отсутствует значение мощности. В конце `df.isnull().sum().sum()` повторно подсчитывает все оставшиеся пропуски.

В выводе будут представлены количество строк до очистки, после неё и число пропущенных значений. Если последнее значение равно нулю, таблица не содержит `NaN` и может использоваться при обучении. Исключённые строки уже не участвуют в последующем анализе.

## 4. Разведочный анализ данных

In [ ]:
df[numeric_columns].hist(figsize=(14, 10), bins=20)
plt.suptitle("Распределения числовых признаков", y=1.02)
plt.tight_layout()
plt.show()

Каждый небольшой график — гистограмма одного признака. По горизонтали отложены значения признака, по вертикали — сколько автомобилей попало в соответствующий интервал.

По форме гистограммы можно увидеть, где находится основная масса наблюдений, есть ли перекос распределения, несколько групп значений или длинные «хвосты».

Перед обучением модели важно понимать структуру данных и замечать необычные распределения.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="weight", y="mpg")
plt.title("Зависимость MPG от веса автомобиля")
plt.xlabel("Вес автомобиля")
plt.ylabel("MPG")
plt.show()

Каждая точка — один автомобиль. По оси X указан вес, по оси Y — его `mpg`.

Облако точек имеет выраженное направление вниз: при увеличении веса `mpg` обычно уменьшается.

Между весом и топливной экономичностью наблюдается заметная отрицательная связь, поэтому `weight` является полезным признаком для линейной регрессии.

На диаграмме показана связь, а не доказанная причинная зависимость. На экономичность могут влиять и другие характеристики автомобиля.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="horsepower", y="mpg")
plt.title("Зависимость MPG от мощности")
plt.xlabel("Мощность")
plt.ylabel("MPG")
plt.show()

Каждая точка — автомобиль. По оси X расположена мощность двигателя, по оси Y — `mpg`.

В среднем более мощные автомобили имеют меньший `mpg`, то есть расходуют больше топлива на одну милю пробега.

`horsepower` также имеет заметную отрицательную связь с целевой переменной.

Для интерпретации учитывается, что мощность связана с массой и объёмом двигателя, поэтому влияние одного признака нельзя полностью отделить по двумерному графику.

In [ ]:
corr = df[numeric_columns].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Матрица корреляции")
plt.show()

Матрица корреляции отражает силу линейной связи между каждой парой числовых признаков. Значения находятся в диапазоне от `-1` до `1`.

- значение близко к `1` — сильная положительная связь;
- близко к `-1` — сильная отрицательная связь;
- около `0` — выраженной линейной связи почти нет.

На главной диагонали всегда стоит `1`, потому что каждый признак полностью коррелирует сам с собой.

Для линейной регрессии особенно полезно увидеть, какие признаки сильнее связаны с `mpg`, а также какие входные признаки сильно коррелируют между собой.

Коэффициенты корреляции рассчитаны по очищенным числовым признакам. Сильная корреляция между двумя входными параметрами может осложнять интерпретацию отдельных коэффициентов регрессии.

### Скрипичная диаграмма: распределение MPG по регионам

In [ ]:
plt.figure(figsize=(9, 5))
sns.violinplot(data=df, x="origin", y="mpg", inner="quartile", cut=0, color=sns.color_palette()[0])
plt.xlabel("Регион производства автомобиля")
plt.ylabel("Топливная экономичность, миль/галлон (MPG)")
plt.title("Скрипичная диаграмма: распределение MPG по регионам")
plt.tight_layout()
plt.show()

Была построена скрипичная диаграмма (*violin plot*). `sns.violinplot` строит сглаженную плотность распределения в каждой категории, `inner="quartile"` отображает квартильные линии, а `cut=0` не продолжает плотность за наблюдаемый диапазон.

По оси X показан регион (`usa`, `europe`, `japan`), по Y — MPG. Чем шире силуэт на определённой высоте, тем больше там относительная концентрация автомобилей. Линии внутри показывают квартильные ориентиры.

Группы могут различаться по распределению MPG. Это **описательное сравнение**, а не доказательство, что страна производства сама по себе вызывает изменение расхода: на него также влияют масса, мощность, год и другие факторы.

### Шестиугольная карта плотности: масса и MPG

In [ ]:
plt.figure(figsize=(8, 5))
h = plt.hexbin(df["weight"], df["mpg"], gridsize=25, mincnt=1, cmap="viridis")
plt.colorbar(h, label="Число автомобилей в ячейке")
plt.xlabel("Масса автомобиля, фунты (lb)")
plt.ylabel("Топливная экономичность, миль/галлон (MPG)")
plt.title("Шестиугольная карта плотности: масса и MPG")
plt.tight_layout()
plt.show()

Была построена двумерная диаграмма частот (*hexbin plot*). `plt.hexbin` разбивает плоскость «масса–MPG» на шестиугольные ячейки; цвет показывает, сколько автомобилей попало в ячейку. `mincnt=1` скрывает пустые области.

По X — масса в фунтах, по Y — MPG, цветовая шкала — количество машин. В отличие от обычного scatter, здесь легче увидеть наиболее населённые области, даже если многие точки перекрываются.

Если более тяжёлые автомобили сосредоточены при меньшем MPG, это поддерживает наблюдение об отрицательной статистической связи. График не даёт причинного вывода.

### Линейный график: средняя экономичность по модельным годам

In [ ]:
by_year = df.groupby("model_year", as_index=False)["mpg"].mean()
plt.figure(figsize=(9, 5))
plt.plot(by_year["model_year"], by_year["mpg"], marker="o")
plt.xlabel("Модельный год (1970–1982)")
plt.ylabel("Средний MPG, миль/галлон")
plt.title("Линейный график: средняя экономичность по модельным годам")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Был построен линейный график (*line plot*) агрегированных данных. `groupby("model_year").mean()` рассчитывает среднее MPG отдельно для каждого модельного года; `plt.plot` соединяет точки в хронологическом порядке.

По X — модельный год, по Y — среднее MPG автомобилей этого года в датасете. Наклон линии отражает изменение групповых средних, **не прогноз отдельного автомобиля**.

Если линия повышается, автомобили более поздних модельных годов в данной выборке в среднем экономичнее. Это не доказывает, что сам год является причиной: одновременно меняются конструкции и состав выборки.

### Групповой boxplot: MPG по числу цилиндров

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="cylinders", y="mpg", order=sorted(df["cylinders"].unique()), color=sns.color_palette()[0])
plt.xlabel("Число цилиндров двигателя")
plt.ylabel("Топливная экономичность, MPG")
plt.title("Групповой boxplot: MPG по числу цилиндров")
plt.tight_layout()
plt.show()

Были построены групповые диаграммы размаха (*grouped boxplots*). `sns.boxplot` строит распределение MPG отдельно для каждой категории числа цилиндров, а `order` упорядочивает категории.

X — число цилиндров, Y — MPG; линия в каждой коробке — медиана, края — квартили, точки вне усов — потенциальные выбросы.

Различия медиан показывают различия центральных значений между группами. Перекрытие коробок означает, что группы не разделяются однозначно. Сам график не доказывает причинного влияния цилиндров.

## 5. Подготовка признаков

In [ ]:
model_data = df.drop(columns=["car_name"]).copy()

# origin — категориальный признак региона (usa/europe/japan), а не числовая величина.
model_data = pd.get_dummies(
    model_data,
    columns=["origin"],
    prefix="origin",
    drop_first=True,
    dtype=int
)

X = model_data.drop(columns=["mpg"])
y = model_data["mpg"]

print("Признаки:")
print(list(X.columns))
print("Размер X:", X.shape)

Для линейной регрессии была подготовлена матрица входных признаков `X` и целевой столбец `y`. Название автомобиля `car_name` удалено, поскольку представляет собой текстовый идентификатор. Категория `origin` преобразована с помощью `pd.get_dummies`: вместо одного текстового столбца создаются бинарные признаки региона производства. Параметр `drop_first=True` оставляет одну категорию базовой.

`X` содержит характеристики автомобиля, по которым будет предсказываться экономичность, а `y = model_data["mpg"]` — реальные значения MPG. Напечатанный список признаков показывает, какие столбцы вошли в модель; размер `X` позволяет проверить количество объектов и итоговое число признаков.

## 6. Разделение выборки

Выборка разделена на две части: 80% объектов для обучения и 20% для независимой оценки качества.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Подготовленные данные были разделены функцией `train_test_split` на обучающую и тестовую части. Параметр `test_size=0.2` выделяет 20% автомобилей для проверки, остальные 80% остаются на обучение. `random_state=42` фиксирует случайное разбиение для повторяемости результатов.

Строки `Train:` и `Test:` выводят размеры матриц. Число столбцов-признаков в обеих частях должно совпадать; различается только количество автомобилей. Тестовые значения не передаются в `fit()` и используются для оценки уже обученной модели.

## 7. Обучение модели

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

Была создана модель `LinearRegression()`. При вызове `model.fit(X_train, y_train)` были определены коэффициенты линейной зависимости между характеристиками автомобиля и MPG. После этого `model.predict(X_test)` рассчитывает прогноз для каждого тестового объекта и сохраняет его в массив `y_pred`.

Эта ячейка ничего не печатает. Её результат — обученный объект `model` и массив предсказанных значений; их сравнение с `y_test` проводится ниже.

## 8. Оценка качества

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")

Качество линейной регрессии оценивается тремя метриками. `mean_absolute_error` вычисляет MAE — среднюю абсолютную ошибку в MPG. `np.sqrt(mean_squared_error(...))` даёт RMSE, который сильнее реагирует на большие промахи. `r2_score` рассчитывает коэффициент детерминации R².

После запуска выводятся три числа. Для MAE и RMSE меньшие значения предпочтительнее; заметное превышение RMSE над MAE может говорить о крупных отдельных ошибках. Для R² результат ближе к единице обычно лучше, около нуля соответствует уровню простого прогноза средним, а отрицательное значение хуже такого прогноза. Численные оценки рассматриваются именно на тестовой выборке.

In [ ]:
results = pd.DataFrame({
    "Фактическое MPG": y_test.to_numpy(),
    "Предсказанное MPG": y_pred
})
results["Абсолютная ошибка"] = (
    results["Фактическое MPG"] - results["Предсказанное MPG"]
).abs()

results.head(10)

Для более предметной проверки была составлена таблица `results`. В неё занесены фактические значения `y_test`, прогнозы `y_pred` и модуль их разницы, рассчитанный методом `.abs()`. В конце `head(10)` оставляет первые десять тестовых объектов.

В таблице каждая строка относится к одному автомобилю. Например, при фактическом MPG 25 и прогнозе 23,8 абсолютная ошибка составит 1,2 MPG. Здесь видны конкретные удачные и неудачные предсказания, которые нельзя определить по одной средней метрике.

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.7)

minimum = min(y_test.min(), y_pred.min())
maximum = max(y_test.max(), y_pred.max())
plt.plot([minimum, maximum], [minimum, maximum], "--")

plt.xlabel("Фактическое MPG")
plt.ylabel("Предсказанное MPG")
plt.title("Фактические и предсказанные значения")
plt.show()

По оси X находятся реальные значения `mpg`, а по оси Y — прогнозы модели. Пунктирная диагональ соответствует идеальному прогнозу, где предсказание полностью совпадает с фактом.

Чем ближе точки к диагонали, тем точнее работает модель. Точки далеко от линии — случаи с большой ошибкой.

Линия равенства задаёт условие идеального прогноза. Расстояние точки от линии по вертикали показывает величину ошибки для соответствующего автомобиля.

In [ ]:
residuals = y_test.to_numpy() - y_pred

plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.7)
plt.axhline(0, linestyle="--")
plt.xlabel("Предсказанное MPG")
plt.ylabel("Ошибка")
plt.title("Остатки модели")
plt.show()

Остаток — это `фактическое значение − предсказание`. Горизонтальная пунктирная линия соответствует нулевой ошибке.

Точки выше нуля означают, что модель занизила MPG; ниже нуля — завысила. Для хорошей линейной модели желательно видеть примерно случайное облако точек вокруг нуля без выраженного рисунка.

Наличие устойчивого изгиба или увеличения разброса при росте прогноза означало бы, что простой линейной формулы недостаточно для описания всей структуры ошибок.

### Гистограмма и KDE: распределение остатков модели

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(residuals, bins=18, kde=True, color=sns.color_palette()[0])
plt.axvline(0, ls="--", color="black")
plt.xlabel("Остаток = фактический MPG − прогноз MPG")
plt.ylabel("Количество автомобилей")
plt.title("Гистограмма и KDE: распределение остатков модели")
plt.tight_layout()
plt.show()

Для сравнения использован график: гистограмма остатков с оценкой плотности (*histogram + kernel density estimate, KDE*). `sns.histplot(..., kde=True)` рисует частоты ошибок и сглаженную форму распределения, вертикальная линия отмечает нулевую ошибку.

X — подписанная ошибка в MPG, Y — число автомобилей в интервале. Центрирование вокруг нуля означает отсутствие сильного среднего смещения, широкие хвосты — наличие крупных ошибок.

Асимметрия или тяжёлые хвосты — повод подробнее изучить ограничения модели. KDE — лишь сглаженное описание данных, а не доказательство нормальности.

### Дополнительная визуализация

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
stats.probplot(residuals, dist="norm", plot=ax)
ax.set_title("Нормальный Q–Q график остатков")
ax.set_xlabel("Теоретические квантили нормального распределения")
ax.set_ylabel("Наблюдаемые квантили остатков (MPG)")
plt.tight_layout()
plt.show()

Для сравнения использован график: квантиль-квантильная диаграмма (*normal Q–Q plot*). `stats.probplot` сравнивает упорядоченные остатки с квантилями теоретического нормального распределения и строит линию сравнения.

X — ожидаемые квантили нормального распределения, Y — квантили реальных ошибок. Если точки примерно вдоль прямой, нормальная модель распределения остатков выглядит правдоподобнее.

Заметный изгиб или сильные отклонения на краях указывают на ненормальные хвосты или асимметрию. **Для получения прогнозов методом OLS нормальность остатков не обязательна**; эта проверка важнее для классических статистических предположений и интервалов.

### Сравнение средней ошибки модели по регионам

In [ ]:
error_groups = pd.DataFrame({
    "origin": df.loc[y_test.index, "origin"].to_numpy(),
    "absolute_error": np.abs(y_test.to_numpy() - y_pred)
})
mean_err = error_groups.groupby("origin")["absolute_error"].mean().sort_values()
plt.figure(figsize=(8, 5))
plt.barh(mean_err.index, mean_err.values)
plt.xlabel("Средняя абсолютная ошибка (MAE), MPG")
plt.ylabel("Регион автомобиля")
plt.title("Сравнение средней ошибки модели по регионам")
plt.tight_layout()
plt.show()

Для сравнения использован график: горизонтальная столбчатая диаграмма (*horizontal bar chart*). `df.loc[y_test.index]` восстанавливает регион каждого **тестового** автомобиля, `abs` считает абсолютную ошибку, `groupby(...).mean()` — её среднее по регионам.

Длина каждого столбца — средняя абсолютная ошибка в MPG среди автомобилей данной группы **в тестовой выборке**. Меньше — точнее.

Различия помогают обнаружить неодинаковую точность в подгруппах. При малом числе тестовых машин в группе оценка может быть нестабильной; это описание, а не основание утверждать о системной дискриминации.

### Эмпирическая функция распределения абсолютной ошибки (ECDF)

In [ ]:
ordered_errors = np.sort(np.abs(residuals))
cumulative_share = np.arange(1, len(ordered_errors) + 1) / len(ordered_errors)
plt.figure(figsize=(8, 5))
plt.step(ordered_errors, cumulative_share, where="post")
plt.axhline(0.9, ls="--", color="gray", label="90% тестовых объектов")
plt.xlabel("Абсолютная ошибка, MPG")
plt.ylabel("Доля объектов с ошибкой не больше X")
plt.title("Эмпирическая функция распределения абсолютной ошибки (ECDF)")
plt.ylim(0, 1.02)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

Была построена эмпирическая функция распределения (*empirical cumulative distribution function, ECDF*). `np.sort` упорядочивает абсолютные ошибки, а `np.arange/n` вычисляет накопленную долю тестовых объектов.

X — величина абсолютной ошибки в MPG, Y — доля автомобилей с ошибкой **не больше** этой величины. Например, точка на уровне `0.9` показывает порог ошибки, в который укладываются 90% тестовых наблюдений.

Кривая, растущая ближе к левому краю, означает более частые небольшие ошибки. Пунктир 90% помогает оценить типичный верхний порог ошибок, а не среднее MAE.

In [ ]:
coefficients = pd.DataFrame({
    "Признак": X.columns,
    "Коэффициент": model.coef_
})
coefficients["Модуль коэффициента"] = coefficients["Коэффициент"].abs()
coefficients.sort_values("Модуль коэффициента", ascending=False)

После обучения были выведены коэффициенты `model.coef_` для каждого входного признака. Таблица связывает название признака с коэффициентом, а дополнительный столбец показывает его модуль. Сортировка по модулю выводит наиболее крупные коэффициенты первыми.

Положительный знак означает, что при увеличении данного признака и неизменности остальных модель повышает прогноз MPG; отрицательный — снижает. При этом масштабы признаков различаются, поэтому по одним абсолютным коэффициентам нельзя делать строгий вывод о сравнительной важности признаков без учёта их единиц измерения.

## Вывод

В работе была обучена линейная регрессия для прогнозирования `mpg`. Перед обучением данные были проверены на пропуски, дубликаты и выбросы, после чего проведен разведочный анализ. Качество модели оценивается по MAE, RMSE и R². По графику фактических и предсказанных значений можно увидеть, насколько близко модель восстанавливает MPG на тестовой выборке. Линейная регрессия дает понятную базовую модель, но реальные зависимости между характеристиками автомобиля не обязаны быть полностью линейными.